# 05 · Control PID dinámico del cristalizador A3110: lazo abierto → lazo cerrado → cambios de consigna

**Orden de trabajo (según lo pedido):**

| Parte | Qué se hace | Para qué |
|---|---|---|
| **1. Lazo abierto** | Se fijan las entradas manipulables en su valor nominal y se aplican perturbaciones una a una | Comprobar que el modelo responde con el signo y la magnitud físicos *antes* de poner control |
| **2. Lazo cerrado** | Mismas perturbaciones, con los lazos `T` (vacío, `PIRCSAHL 3101`), `V` (nivel, `LIRCSAHL 3101`) y dilución en razón (`FFIC 3102` → `FIRC 3103`) | Ver cuánto atenúa el control cada perturbación y a qué costo (variables manipuladas) |
| **3. Cambios de consigna** | Escalones y rampa en `T` y `V`; barrido del parámetro de filtro ε (IMC) y ponderación de consigna β | Ver seguimiento, sobrepaso, tiempo de asentamiento y saturación |
| **B. Agua de dilución** | Sin cambios de fondo: el análisis previo de `AIT 3105 → FFIC 3102 → FIRC 3103` | Se conserva tal cual |

**Ecuaciones de las sesiones usadas aquí** (apuntes de control, sesiones 16–22):

* PI/PID de posición: $u = \bar u + K_c\,[e + \tfrac{1}{\tau_I}\int e\,dt + \tau_D\,\dot e]$ (sesiones 16–17). Offset nulo con acción integral.
* Proceso aproximado por **FOPDT** $G(s)=K e^{-\theta s}/(\tau s+1)$ identificado con un escalón (sesión 18).
* **IMC** → PI (sesiones 21–22): $K_c=\dfrac{\tau+\theta/2}{K(\varepsilon+\theta/2)}$, $\tau_I=\tau+\theta/2$. Un solo parámetro de ajuste ε (constante de tiempo de lazo cerrado). Con θ≈0 coincide con SIMC con $\tau_c=\varepsilon$ (el que usa `autotune`).
* Cascada y *feedforward* (sesión 19): el lazo `FIRC 3103` es el esclavo rápido; la razón `FFIC 3102` compensa la perturbación medida de caudal de alimentación.
* Criterio de desempeño: IAE $=\int|e|\,dt$, sobrepaso y tiempo de asentamiento (±5 %), como en los talleres 18 y 21.

**Qué se modifica del código original.** Se agregan las partes 1–3; el código de la parte B no cambia. El controlador `PlantController` y el modelo `Plant` no se tocaron: los cambios de consigna se hacen con un envoltorio (`ConsignaVariable`) que reescribe `ctrl.sp` en cada paso.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # raíz del repo (donde está msmpr_core)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params, placeholders
from msmpr_core.plant import Plant, simulate
from msmpr_core import control as C
os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
q0, qs0 = u0["Q_dil_cmd"], d0["Q_stock"]
R0 = q0 / qs0
DT = 5.0
tun, info = C.autotune(pl, x0, u0, d0, dt_s=DT)
sp = {"T": x0[1], "V": x0[0], "rho": 0.0}
print(f"Dilución nominal {3600*q0:.2f} m3/h; razón nominal R0 = {R0:.4f}; ppm nominal {m0['ppm_ClO4']:.1f}; recuperación {m0['recovery_pct']:.1f} %")
print(f"T = {x0[1]:.1f} °C, V = {x0[0]:.0f} m3, Q_out = {3600*u0['Q_out']:.0f} m3/h, m_evap = {u0['m_evap']:.3f} kg/s, tau_res = {m0['tau_h']:.2f} h")
print("Sintonía (autotune):")
for k in ("T", "V", "flow"):
    t_ = tun[k]; Ti = t_.Kp / t_.Ki if t_.Ki > 0 else np.inf
    print(f"  {k:5s} Kp = {t_.Kp:9.4g}  Ti = {Ti:8.0f} s  acción {'directa' if t_.direct else 'inversa'}")
print("Identificación FOPDT lazo T:", {k: round(float(v), 4) for k, v in info['T'].items() if k != 'unid'})

In [ ]:
# ---------------------------------------------------------------- utilidades de esta versión
class ConsignaVariable:
    # Envoltorio de PlantController: reescribe las consignas ('T', 'V') en cada paso con funciones de t.
    def __init__(self, ctrl, funcs): self.ctrl, self.funcs = ctrl, funcs
    def __call__(self, t, x, plant):
        for k, f in self.funcs.items():
            self.ctrl.sp[k] = f(t)
        return self.ctrl(t, x, plant)

def imc_pi(K, tau, theta, eps):
    # PI por IMC (sesiones 21-22). Devuelve LoopTuning.
    Kp = (tau + theta/2) / (abs(K) * (eps + theta/2)); Ti = tau + theta/2
    return C.LoopTuning(Kp=Kp, Ki=Kp/Ti, Kd=0.0, direct=bool(K < 0))

def metricas_consigna(t_s, y, y0, y1, t_step, band=0.05):
    # Sobrepaso [%], tiempo de asentamiento (+-band del salto) [h] e IAE [unid*h] de una respuesta a escalón.
    s = (y - y0) / (y1 - y0); m = t_s >= t_step
    over = 100*max(s[m].max() - 1.0, 0.0)
    fuera = np.abs(s[m] - 1.0) > band
    tset = (t_s[m][fuera][-1] - t_step)/3600 if fuera.any() else 0.0
    iae = np.trapezoid(np.abs(y[m] - y1), t_s[m]) / 3600
    return over, tset, iae

TP = 3600.0                      # instante de la perturbación o del cambio de consigna [s]
H_OL = 18*3600                   # horizonte de las perturbaciones [s]
d_step = lambda **kw: (lambda t: {**d0, **{k: (f(t) if callable(f) else f) for k, f in kw.items()}})

ESC = {   # nombre: (perturbación, descripción)
    "Q_stock +10 %":   (lambda t: {**d0, "Q_stock": qs0*(1.10 if t >= TP else 1.0)}),
    "T_feed +5 °C":    (lambda t: {**d0, "T_feed": d0["T_feed"] + (5.0 if t >= TP else 0.0)}),
    "ClO4 alim. +25 %":(lambda t: {**d0, "CP_stock": d0["CP_stock"]*(1.25 if t >= TP else 1.0)}),
    "KNO3 alim. +5 %": (lambda t: {**d0, "C_stock": d0["C_stock"]*(1.05 if t >= TP else 1.0)}),
    "T_agua dil. +10 °C": (lambda t: {**d0, "T_dil": d0["T_dil"] + (10.0 if t >= TP else 0.0)}),
    "Válvula dil. -20 %": (lambda t: {**d0, "valve_gain": 0.8 if t >= TP else 1.0}),
}
COL = dict(zip(ESC, ["tab:blue", "tab:red", "tab:green", "tab:purple", "tab:orange", "tab:brown"]))

def lazo_abierto(dist, H=H_OL, dt=DT, log=12):
    return simulate(pl, x0, H, dt, lambda t, x, pl_: u0, disturbance=dist, log_every=log)

def lazo_cerrado(dist, H=H_OL, dt=DT, log=12, tuning=tun, enabled=("T", "V", "ratio"), consigna=None):
    qsf = lambda t: dist(t)["Q_stock"] if dist is not None else qs0
    ctrl = C.PlantController(pl, u0, sp, tuning, enabled=enabled, q_stock_meas=qsf, ratio_nom=R0)
    if consigna: ctrl = ConsignaVariable(ctrl, consigna)
    return simulate(pl, x0, H, dt, ctrl, disturbance=dist, log_every=log), ctrl

## 1. Lazo abierto: ¿el modelo responde bien a las perturbaciones?

Aquí **no hay ningún controlador**: `Q_out`, `m_evap`, `Q_dil_cmd` y `alpha` quedan en su valor nominal. Se hacen cuatro verificaciones.

1. El estado nominal es estacionario (el modelo no se mueve solo).
2. La identificación FOPDT de la temperatura (la base de la sintonía) reproduce la simulación.
3. Cada perturbación mueve las variables en el sentido físico esperado.
4. Dos balances comprobables a mano: la pendiente inicial del volumen y la ganancia de la válvula.

Nota: sin control el volumen es un **integrador** (no tiene realimentación), de modo que una perturbación de caudal lo hace derivar sin límite. Eso es correcto, y es justamente lo que el lazo de nivel corrige.

In [ ]:
# 1.1  Reposo en lazo abierto con entradas y perturbaciones nominales
df0 = lazo_abierto(None, H=6*3600, log=60)
dev = {k: float(np.max(np.abs(df0[k] - df0[k].iloc[0]))) for k in ("V", "T", "C", "CP", "ppm_ClO4", "recovery_pct")}
print("Desviación máxima respecto del estado nominal en 6 h (debe ser ~0):")
print({k: f"{v:.2e}" for k, v in dev.items()})
assert max(dev.values()) < 1e-6, "el estado nominal no es estacionario"
print("OK: el estacionario se sostiene en lazo abierto.")

In [ ]:
# 1.2  Validación de la identificación FOPDT de T frente a m_evap (escalón +10 % en la evaporación)
dmev = 0.10*u0["m_evap"]; K_, tau_, th_ = info["T"]["K"], info["T"]["tau_s"], info["T"]["theta_s"]
t_s, T_sim = C.step_test(pl, x0, u0, d0, "m_evap", dmev, lambda x, u, dd: x[1], 12*3600, dt_s=10.0)
T_fopdt = x0[1] + K_*dmev*(1 - np.exp(-np.clip(t_s - th_, 0, None)/tau_))
rmse = float(np.sqrt(np.mean((T_sim - T_fopdt)**2)))
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(t_s/3600, T_sim, label="modelo no lineal (Plant)"); ax.plot(t_s/3600, T_fopdt, "--", label=f"FOPDT: K={K_:.2f} °C/(kg/s), τ={tau_/3600:.2f} h, θ≈{th_:.0f} s")
ax.set(xlabel="t [h]", ylabel="T [°C]", title=f"Escalón de +10 % en m_evap (RMSE = {rmse:.3f} °C)"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("../docs/fig/05_validacion_FOPDT.png"); plt.show()
print(f"RMSE = {rmse:.3f} °C | cambio final de T: {T_sim[-1]-x0[1]:+.2f} °C (el FOPDT predice {K_*dmev:+.2f} °C; la diferencia es la no linealidad)")

In [ ]:
# 1.3  Perturbaciones en lazo abierto (una a la vez, a t = 1 h)
OL = {n: lazo_abierto(f) for n, f in ESC.items()}
fig, ax = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for n, df in OL.items():
    ax[0, 0].plot(df.t_h, df["T"] - x0[1], color=COL[n], label=n)
    ax[0, 1].plot(df.t_h, df.V - x0[0], color=COL[n])
    ax[1, 0].plot(df.t_h, df.ppm_ClO4, color=COL[n])
    ax[1, 1].plot(df.t_h, df.recovery_pct, color=COL[n])
ax[0, 0].set(ylabel="ΔT [°C]", title="Temperatura"); ax[0, 0].legend(fontsize=7)
ax[0, 1].set(ylabel="ΔV [m³]", title="Volumen (integrador: deriva sin nivel)")
ax[1, 0].set(xlabel="t [h]", ylabel="ppm", title="ClO₄⁻ en producto"); ax[1, 0].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1)
ax[1, 1].set(xlabel="t [h]", ylabel="%", title="Recuperación de KNO₃")
for a in ax.ravel(): a.axvline(TP/3600, color="k", lw=0.6, ls=":")
fig.suptitle("Lazo abierto: sin controlador", y=1.0); fig.tight_layout(); fig.savefig("../docs/fig/05_lazo_abierto.png"); plt.show()

In [ ]:
# Tabla de verificación: sentido esperado (balances y termodinámica) frente a lo simulado
esperado = {   # variable: signo esperado de la variación final (0 = sin efecto)
    "Q_stock +10 %":      {"V": +1, "T": +1, "ppm_ClO4": +1},    # más alimentación tibia (55 °C > 40 °C); entra más ClO4 y la purga no cambia
    "T_feed +5 °C":       {"V": 0,  "T": +1, "ppm_ClO4": 0},
    "ClO4 alim. +25 %":   {"V": 0,  "T": 0,  "ppm_ClO4": +1},
    "KNO3 alim. +5 %":    {"V": 0,  "T": +1, "ppm_ClO4": 0},     # más soluto: más calor de cristalización
    "T_agua dil. +10 °C": {"V": 0,  "T": +1, "ppm_ClO4": 0},
    "Válvula dil. -20 %": {"V": -1, "T": +1, "ppm_ClO4": +1},    # menos agua fría y menos dilución del ClO4
}
rows = []
for n, df in OL.items():
    f = {"V": df.V.iloc[-1] - x0[0], "T": df["T"].iloc[-1] - x0[1], "ppm_ClO4": df.ppm_ClO4.iloc[-1] - m0["ppm_ClO4"]}
    tol = {"V": 1.0, "T": 0.01, "ppm_ClO4": 0.05}
    row = {"perturbación": n}
    ok = True
    for v, s in esperado[n].items():
        sg = 0 if abs(f[v]) < tol[v] else int(np.sign(f[v]))
        ok &= (sg == s)
    row.update({"ΔV final [m3]": f["V"], "ΔT final [°C]": f["T"], "Δppm final": f["ppm_ClO4"],
                "ΔY final [pts]": df.recovery_pct.iloc[-1] - m0["recovery_pct"], "signo físico": "OK" if ok else "REVISAR"})
    rows.append(row)
tab_ol = pd.DataFrame(rows).set_index("perturbación"); tab_ol.round(3)

In [ ]:
# 1.4  Dos balances comprobables sin simular el controlador
# (a) Al subir Q_stock en ΔQ, el volumen debe crecer con pendiente ΔQ (nada más cambia al instante).
df = OL["Q_stock +10 %"]; a = df[(df.t_h*3600 >= TP) & (df.t_h*3600 <= TP + 600)]
pend = (a.V.iloc[-1] - a.V.iloc[0]) / (3600*(a.t_h.iloc[-1] - a.t_h.iloc[0]))
print(f"(a) dV/dt inicial simulada = {pend:.5f} m3/s | ΔQ_stock = {0.10*qs0:.5f} m3/s | error = {100*abs(pend/(0.10*qs0)-1):.2f} %")
assert abs(pend/(0.10*qs0) - 1) < 0.03
# (b) Con la válvula al 80 %, el caudal real de dilución debe converger a 0.8 del comandado
Qd_fin = OL["Válvula dil. -20 %"].Qd.iloc[-1]
print(f"(b) Qd final / Qd comandado = {Qd_fin/q0:.4f} (esperado 0.8000)")
assert abs(Qd_fin/q0 - 0.8) < 1e-3
print("OK: balances de volumen y de válvula consistentes.")

**Lectura de la parte 1.**

* Sin control, el volumen deriva con cualquier desbalance de caudales: es un integrador puro. Con Q_stock +10 % crece cientos de m³ en 18 h; eso no es un error del modelo, es lo que el lazo de nivel debe evitar.
* La temperatura responde con una constante de tiempo de ~2 h (la del FOPDT), por eso el horizonte es largo. Las cinco perturbaciones de proceso la suben: más alimentación tibia (55 °C frente a 40 °C), más temperatura de alimentación o de agua de dilución, más soluto (más calor de cristalización a evacuar) o menos agua fría de dilución (válvula).
* Una subida de T reduce la sobresaturación y con ello la recuperación: con T_feed +5 °C la recuperación cae ~8 puntos, mientras que los ppm casi no se mueven. La excepción es el soluto de alimentación, que sube T pero también la recuperación porque hay más KNO₃ disponible.
* El ClO₄⁻ sólo mueve los ppm, y con la lentitud de la purga α = 0.25: tarda horas en llegar a su nuevo valor (la tabla de la parte 2 lo confirma en 24 h). Menos dilución o más alimentación también lo suben un poco.

## 2. Lazo cerrado: las mismas perturbaciones con el controlador

Lazos activos: `T` (`m_evap`), `V` (`Q_out`) y dilución en razón fija (`FFIC 3102` → `FIRC 3103`: la consigna de dilución es R₀ × caudal de alimentación medido). Se compara con el lazo abierto de la parte 1.

In [ ]:
CL = {n: lazo_cerrado(f)[0] for n, f in ESC.items()}
fig, ax = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for n in ESC:
    o, c = OL[n], CL[n]
    ax[0, 0].plot(o.t_h, o["T"] - x0[1], color=COL[n], ls="--", lw=0.9); ax[0, 0].plot(c.t_h, c["T"] - x0[1], color=COL[n], label=n)
    ax[0, 1].plot(c.t_h, c.V - x0[0], color=COL[n])
    ax[0, 2].plot(c.t_h, c.ppm_ClO4, color=COL[n]); ax[0, 2].plot(o.t_h, o.ppm_ClO4, color=COL[n], ls="--", lw=0.9)
    ax[1, 0].plot(c.t_h, 100*(c.u_m_evap/u0["m_evap"] - 1), color=COL[n])
    ax[1, 1].plot(c.t_h, 100*(c.u_Q_out/u0["Q_out"] - 1), color=COL[n])
    ax[1, 2].plot(c.t_h, 100*(c.Qd/q0 - 1), color=COL[n])
ax[0, 0].set(ylabel="ΔT [°C]", title="T (trazo: lazo abierto; sólido: cerrado)"); ax[0, 0].legend(fontsize=7)
ax[0, 1].set(ylabel="ΔV [m³]", title="V en lazo cerrado (en abierto deriva cientos de m³)")
ax[0, 2].set(ylabel="ppm", title="ClO₄⁻ en producto"); ax[0, 2].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1)
ax[1, 0].set(xlabel="t [h]", ylabel="Δ m_evap [%]", title="Acción de control: vacío/evaporación")
ax[1, 1].set(xlabel="t [h]", ylabel="Δ Q_out [%]", title="Acción de control: descarga")
ax[1, 2].set(xlabel="t [h]", ylabel="Δ Q_dil real [%]", title="Agua de dilución (FIRC 3103)")
for a in ax.ravel(): a.axvline(TP/3600, color="k", lw=0.6, ls=":")
fig.tight_layout(); fig.savefig("../docs/fig/05_lazo_cerrado_perturbaciones.png"); plt.show()

In [ ]:
def iae(t_h, e): return float(np.trapezoid(np.abs(e), t_h))
rows = []
for n in ESC:
    o, c = OL[n], CL[n]
    rows.append({"perturbación": n,
                 "max|ΔT| abierto [°C]": np.abs(o["T"] - x0[1]).max(), "max|ΔT| cerrado [°C]": np.abs(c["T"] - x0[1]).max(),
                 "IAE_T abierto [°C·h]": iae(o.t_h, o["T"] - x0[1]), "IAE_T cerrado [°C·h]": iae(c.t_h, c["T"] - x0[1]),
                 "max|ΔV| abierto [m3]": np.abs(o.V - x0[0]).max(), "max|ΔV| cerrado [m3]": np.abs(c.V - x0[0]).max(),
                 "ppm final abierto": o.ppm_ClO4.iloc[-1], "ppm final cerrado": c.ppm_ClO4.iloc[-1],
                 "Y final abierto [%]": o.recovery_pct.iloc[-1], "Y final cerrado [%]": c.recovery_pct.iloc[-1]})
tab_cl = pd.DataFrame(rows).set_index("perturbación"); tab_cl.round(3)

In [ ]:
# Consistencia modelo-controlador: con acción integral, el estado final en lazo cerrado debe coincidir con el estacionario
# del mismo modelo evaluado en la perturbación (T y V en consigna, dilución = R0 * Q_stock). 24 h para que la purga converja.
rows = []
for n in ("T_feed +5 °C", "KNO3 alim. +5 %", "Q_stock +10 %", "ClO4 alim. +25 %"):
    f = ESC[n]; dfc, _ = lazo_cerrado(f, H=24*3600, log=60)
    d_end = f(24*3600); xs, us, ds = pl.steady_state(alpha=0.25, d=d_end, Q_dil=R0*d_end["Q_stock"]); ms = pl.metrics(xs, us, ds)
    rows.append({"perturbación": n, "ppm simulado": dfc.ppm_ClO4.iloc[-1], "ppm estacionario": ms["ppm_ClO4"],
                 "Y simulada [%]": dfc.recovery_pct.iloc[-1], "Y estacionaria [%]": ms["recovery_pct"],
                 "error relativo Y [%]": 100*abs(dfc.recovery_pct.iloc[-1]/ms["recovery_pct"] - 1)})
cons = pd.DataFrame(rows).set_index("perturbación"); display(cons.round(4)) if "display" in globals() else print(cons.round(4))
assert cons["error relativo Y [%]"].max() < 0.1

**Lectura de la parte 2.**

* Los lazos `T` y `V` reducen mucho el efecto de las perturbaciones sobre la temperatura y el volumen: con T_feed +5 °C el ΔT máximo baja de 3.1 a 0.2 °C, y con Q_stock +10 % el volumen deja de derivar (de 450 m³ a ~3 m³). El estado final cumple offset nulo (sesiones 16–17), como confirma la tabla de consistencia. La recuperación vuelve cerca del nominal (11.0 %) cuando la perturbación es de T o de caudal; con más KNO₃ en la alimentación sube porque hay más soluto, y con Q_stock +10 % baja algo porque la razón fija de dilución añade más agua.
* Los ppm de ClO₄⁻ **no** se corrigen con estos lazos: ni T ni V ni la razón fija ven el ClO₄⁻. Sólo la compensación vía `AIT 3105` (parte B) actúa sobre él.
* La perturbación de la válvula la absorbe `FIRC 3103` (esclavo rápido de la cascada), por eso casi no aparece en T.
* La tabla de consistencia confirma que el modelo dinámico y el estacionario son una sola fuente de ecuaciones: el estado al que llega el lazo cerrado es el estacionario esperado.

## 3. Cambios de consigna (seguimiento)

Se mueven las consignas de `T` y de `V` sin perturbaciones. El seguimiento se mide con sobrepaso, tiempo de asentamiento (±5 % del salto) e IAE. Además se compara la sintonía automática con la IMC de las sesiones 21–22 para distintos ε, y el efecto de la ponderación de consigna β sobre la acción de control.

In [ ]:
# 3.1  Consigna de temperatura: escalones de -2 y +2 °C y rampa de -2 °C/h entre 1 h y 2 h
H_SP = 8*3600
sp_T = {"escalón -2 °C": lambda t: x0[1] - (2.0 if t >= TP else 0.0),
        "escalón +2 °C": lambda t: x0[1] + (2.0 if t >= TP else 0.0),
        "rampa -2 °C/h (1 h)": lambda t: x0[1] - 2.0*np.clip((t - TP)/3600.0, 0.0, 1.0)}
SPT = {n: lazo_cerrado(None, H=H_SP, consigna={"T": f})[0] for n, f in sp_T.items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for (n, df), col in zip(SPT.items(), ["tab:blue", "tab:red", "tab:green"]):
    ts = df.t_h.values*3600
    ax[0].plot(df.t_h, df["T"], color=col, label=n); ax[0].plot(df.t_h, [sp_T[n](t) for t in ts], color=col, ls=":", lw=0.9)
    ax[1].plot(df.t_h, df.u_m_evap, color=col)
    ax[2].plot(df.t_h, df.recovery_pct, color=col)
ax[0].set(xlabel="t [h]", ylabel="T [°C]", title="PV (sólido) y consigna (punteado)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="t [h]", ylabel="m_evap [kg/s]", title="Variable manipulada (límites 0.2x y 3x nominal)")
ax[1].axhline(3.0*u0["m_evap"], color="r", lw=0.7); ax[1].axhline(0.2*u0["m_evap"], color="r", lw=0.7)
ax[2].set(xlabel="t [h]", ylabel="Y [%]", title="Recuperación de KNO₃")
fig.tight_layout(); fig.savefig("../docs/fig/05_consigna_T.png"); plt.show()
rows = []
for n in ("escalón -2 °C", "escalón +2 °C"):
    df = SPT[n]; y1 = x0[1] + (-2.0 if "-2" in n else 2.0)
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], y1, TP)
    rows.append({"caso": n, "sobrepaso [%]": o, "t asentamiento ±5 % [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max(),
                 "m_evap mín [kg/s]": df.u_m_evap.min(), "ppm final": df.ppm_ClO4.iloc[-1], "Y final [%]": df.recovery_pct.iloc[-1]})
df = SPT["rampa -2 °C/h (1 h)"]; e = df["T"].values - np.array([sp_T["rampa -2 °C/h (1 h)"](t) for t in df.t_h.values*3600])
rows.append({"caso": "rampa -2 °C/h", "sobrepaso [%]": np.nan, "t asentamiento ±5 % [h]": np.nan, "IAE [°C·h]": iae(df.t_h, e),
             "m_evap máx [kg/s]": df.u_m_evap.max(), "m_evap mín [kg/s]": df.u_m_evap.min(), "ppm final": df.ppm_ClO4.iloc[-1], "Y final [%]": df.recovery_pct.iloc[-1]})
print(f"Error máximo de seguimiento en la rampa: {np.abs(e).max():.3f} °C")
tab_spT = pd.DataFrame(rows).set_index("caso"); tab_spT.round(3)

In [ ]:
# 3.2  Efecto de ε (IMC) y de la ponderación de consigna β sobre el escalón de -2 °C
K_, tau_, th_ = info["T"]["K"], info["T"]["tau_s"], info["T"]["theta_s"]
cand = {"autotune (SIMC, ε=0.1τ)": tun["T"]}
cand.update({f"IMC ε={f}τ": imc_pi(K_, tau_, th_, f*tau_) for f in (0.05, 0.3, 1.0)})
fT = sp_T["escalón -2 °C"]; rows = []; fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for (n, tt) in cand.items():
    df, _ = lazo_cerrado(None, H=H_SP, tuning={**tun, "T": tt}, consigna={"T": fT})
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], x0[1] - 2.0, TP)
    ax[0].plot(df.t_h, df["T"], label=n); ax[1].plot(df.t_h, df.u_m_evap)
    rows.append({"sintonía": n, "Kp": tt.Kp, "Ti [s]": tt.Kp/tt.Ki, "sobrepaso [%]": o, "t asent. [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max()})
ax[0].plot(df.t_h, [fT(t) for t in df.t_h*3600], "k:", lw=0.9, label="consigna"); ax[0].legend(fontsize=7)
ax[0].set(xlabel="t [h]", ylabel="T [°C]", title="Escalón de consigna -2 °C: barrido de ε")
ax[1].axhline(3.0*u0["m_evap"], color="r", lw=0.7); ax[1].set(xlabel="t [h]", ylabel="m_evap [kg/s]", title="Variable manipulada (saturación en rojo)")
fig.tight_layout(); fig.savefig("../docs/fig/05_barrido_epsilon.png"); plt.show()
tab_eps = pd.DataFrame(rows).set_index("sintonía"); display(tab_eps.round(3)) if "display" in globals() else print(tab_eps.round(3))

rows = []
for beta in (1.0, 0.5, 0.0):
    qsf = lambda t: qs0
    ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio"), q_stock_meas=qsf, ratio_nom=R0); ctrl.pid_T.beta = beta
    ctrl.pid_T.I = -ctrl.pid_T.Kp*ctrl.pid_T.s*(beta*x0[1] - x0[1])      # arranque sin salto: con β≠1, P(SP=PV) = Kp(β-1)SP ≠ 0
    df = simulate(pl, x0, H_SP, DT, ConsignaVariable(ctrl, {"T": fT}), log_every=12)
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], x0[1] - 2.0, TP)
    rows.append({"β": beta, "sobrepaso [%]": o, "t asent. [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max()})
print("Ponderación de consigna β (P actúa sobre β·SP - PV; el integral siempre sobre SP - PV). El integral se precarga para que el estacionario inicial no se perturbe:")
pd.DataFrame(rows).set_index("β").round(3)

In [ ]:
# 3.3  Consigna de nivel: escalones de ±20 m3 (±3 %) y rampa de +20 m3 en 1 h
sp_V = {"escalón +20 m3": lambda t: x0[0] + (20.0 if t >= TP else 0.0),
        "escalón -20 m3": lambda t: x0[0] - (20.0 if t >= TP else 0.0),
        "rampa +20 m3/h": lambda t: x0[0] + 20.0*np.clip((t - TP)/3600.0, 0.0, 1.0)}
SPV = {n: lazo_cerrado(None, H=H_SP, consigna={"V": f})[0] for n, f in sp_V.items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); rows = []
for (n, df), col in zip(SPV.items(), ["tab:blue", "tab:red", "tab:green"]):
    ts = df.t_h.values*3600; ref = np.array([sp_V[n](t) for t in ts])
    ax[0].plot(df.t_h, df.V, color=col, label=n); ax[0].plot(df.t_h, ref, color=col, ls=":", lw=0.9)
    ax[1].plot(df.t_h, df.u_Q_out*3600, color=col); ax[2].plot(df.t_h, df["T"] - x0[1], color=col)
    if "escalón" in n:
        o, tset, ia = metricas_consigna(ts, df.V.values, x0[0], ref[-1], TP)
    else:
        o, tset, ia = np.nan, np.nan, iae(df.t_h, df.V.values - ref)
    rows.append({"caso": n, "sobrepaso [%]": o, "t asent. [h]": tset, "IAE [m3·h]": ia, "Q_out mín [m3/h]": 3600*df.u_Q_out.min(),
                 "Q_out máx [m3/h]": 3600*df.u_Q_out.max(), "max|ΔT| [°C]": np.abs(df["T"] - x0[1]).max()})
ax[0].set(xlabel="t [h]", ylabel="V [m³]", title="Nivel: PV (sólido) y consigna (punteado)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="t [h]", ylabel="Q_out [m³/h]", title="Variable manipulada"); ax[2].set(xlabel="t [h]", ylabel="ΔT [°C]", title="Interacción sobre la temperatura")
fig.tight_layout(); fig.savefig("../docs/fig/05_consigna_V.png"); plt.show()
tab_spV = pd.DataFrame(rows).set_index("caso"); tab_spV.round(3)

**Lectura de la parte 3.**

* **Temperatura.** Ambos escalones saturan la variable manipulada un rato: el de -2 °C llega al máximo (3× nominal) y el de +2 °C al mínimo (0.2× nominal), porque el término proporcional entrega de golpe Kp·ΔSP. El anti-windup por retrocálculo evita que el integral se descontrole, pero el sobrepaso (≈5 % y ≈2 %) y el tiempo de asentamiento sí se resienten. La respuesta es asimétrica (≈2.0 h para -2 °C frente a ≈0.5 h para +2 °C): es el efecto de la no linealidad del proceso (la cinética y el calor de cristalización dependen de T).
* **Barrido de ε.** ε chico (lazo rápido) da más ganancia, menor IAE y más esfuerzo de control (con ε = 0.05τ el actuador sigue saturando); ε = 0.3τ ya no satura y ε = τ no tiene sobrepaso pero tarda ≈6 h. Es el compromiso de la sesión 21: un solo parámetro para negociar velocidad contra robustez y uso del actuador. La sintonía automática usa SIMC con Ti = 4τ_c (más acción integral que IMC con Ti = τ), lo que explica su sobrepaso algo mayor.
* **Ponderación β.** Con β = 0.5 el golpe proporcional se reduce a la mitad: el actuador llega sólo a ≈3.3 kg/s (sin saturar), el sobrepaso baja a <1 % y el IAE sube. Con β = 0 el actuador apenas se mueve al principio y la respuesta es mucho más lenta. Detalle de implementación: `PID` arranca con integral nulo, de modo que con β ≠ 1 hay que precargarlo (como se hace en el código) para que el estacionario inicial no se perturbe.
* **Rampa.** Un PI sigue una rampa con error de seguimiento no nulo mientras dura la rampa (el error máximo está en la salida) y lo elimina después, como corresponde a un lazo tipo 1.
* **Nivel.** El lazo `V` es integrador con τ_c = 600 s y presenta sobrepaso de ~13 % ante escalones de consigna (típico de PI sobre integrador con Ti = 4τ_c). Para evitarlo en planta conviene una rampa de consigna o β < 1 en el nivel. La interacción sobre T es mínima (centésimas de °C).
* Cambiar las consignas de T o V cambia el punto de operación: la recuperación y los ppm finales de la tabla ya no son los nominales, y eso se debe analizar con la economía del proceso, no sólo con el desempeño del lazo.

---
# Parte B · Agua de dilución: `AIT 3105 → FFIC 3102 → FIRC 3103` (sin cambios)
El resto del notebook es el análisis original, que no se modificó salvo la numeración de los títulos y la reutilización de las variables definidas arriba (`p`, `pl`, `x0`, `u0`, `d0`, `q0`, `qs0`, `R0`, `DT`, `tun`, `sp`).

## B1. ¿Cuánta agua haría falta para compensar un aumento de ClO₄⁻? (estático)
Para mantener la mezcla en su concentración nominal, la dilución debe crecer con el ClO₄⁻ de entrada. Se resuelve el estacionario con y sin esa compensación.

In [ ]:
rows = []
for f in (1.05, 1.10, 1.25, 1.50):
    R = f*(1 + R0) - 1; Qd = R*qs0
    d = {**d0, "CP_stock": f*d0["CP_stock"]}
    xs, us, ds = pl.steady_state(alpha=0.25, d=d); ms = pl.metrics(xs, us, ds)                   # sin compensar
    xc, uc, dc = pl.steady_state(alpha=0.25, d=d, Q_dil=Qd); mc = pl.metrics(xc, uc, dc)       # con compensación total
    rows.append({"ClO4 de entrada x": f, "dilución necesaria [m3/h]": 3600*Qd, "veces la nominal": Qd/q0,
                 "ppm sin comp.": ms["ppm_ClO4"], "Y sin comp. %": ms["recovery_pct"],
                 "ppm con comp.": mc["ppm_ClO4"], "Y con comp. %": mc["recovery_pct"]})
req = pd.DataFrame(rows).set_index("ClO4 de entrada x"); req.round(2)

**Lectura.** Compensar el ClO₄⁻ con agua es caro: la dilución necesaria crece varias veces sobre la nominal y la recuperación de KNO₃ cae rápido, porque el agua baja la concentración de la alimentación hacia la solubilidad. Con los placeholders actuales (alimentación apenas sobre la solubilidad supuesta) el efecto es extremo; la **tendencia** (recuperación contra pureza) es lo que importa, y el tamaño de la válvula de dilución pasa a ser una restricción de diseño.

## B2. Etapa 1 · `FIRC 3103` solo
Consigna +15 % a los 10 min y pérdida de 20 % de ganancia de la válvula a los 30 min (ruido de medición 1 %). Muestreo de 2 s.

In [ ]:
tun2, _ = C.autotune(pl, x0, u0, d0, dt_s=2.0)
flow_sp = lambda t: q0*(1.15 if t >= 600 else 1.0)
dist1 = lambda t: {**d0, "valve_gain": 0.8 if t >= 1800 else 1.0}
cl = simulate(pl, x0, 3600, 2.0, C.PlantController(pl, u0, sp, tun2, enabled=("flow",), flow_sp=flow_sp, noise_rel={"flow": 0.01}, seed=1), disturbance=dist1, log_every=5)
ol = simulate(pl, x0, 3600, 2.0, lambda t, x, pl_: {**u0, "Q_dil_cmd": flow_sp(t)}, disturbance=dist1, log_every=5)
late = lambda df: 100*abs(df.Qd.values[df.t_h.values > 0.9].mean()/(1.15*q0) - 1)
print(f"Error final tras la perturbación de válvula: FIRC 3103 {late(cl):.2f} %  |  válvula manual {late(ol):.2f} %")

## B3. Etapa 2 · `FFIC 3102`: la dilución sigue al caudal de alimentación
La alimentación sube +12 % y +6 °C a las 2 h. Con razón fija, el caudal de dilución sube con ella.

In [ ]:
Qs = lambda t: qs0*(1.12 if t >= 2*3600 else 1.0)
dist2 = lambda t: {**d0, "Q_stock": Qs(t), "T_feed": d0["T_feed"] + (6.0 if t >= 2*3600 else 0.0)}
H = 8*3600
fija = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow")), disturbance=dist2, log_every=12)
razon = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio"), q_stock_meas=Qs, ratio_nom=R0), disturbance=dist2, log_every=12)
pd.DataFrame({"dilución final [m3/h]": {"consigna fija": 3600*fija.Qd.iloc[-1], "razón fija (FFIC 3102)": 3600*razon.Qd.iloc[-1]},
              "razón dilución/alimentación": {"consigna fija": fija.Qd.iloc[-1]/Qs(H), "razón fija (FFIC 3102)": razon.Qd.iloc[-1]/Qs(H)}}).round(4)

## B4. Etapa 3 · `AIT 3105`: pulso de ClO₄⁻ y retardo del analizador
Pulso de **×1.25 en el ClO₄⁻ de alimentación entre las 3 h y las 5 h**. Como el retardo del analizador aún no está definido, se barre entre 0 y 2 h. Además: válvula de dilución limitada a 3× la nominal frente a una capaz de 8×, y compensación parcial (`gain` = 0.3).

In [ ]:
def pulso(t): return d0["CP_stock"]*(1.25 if 3*3600 <= t < 5*3600 else 1.0)
dist3 = lambda t: {**d0, "CP_stock": pulso(t)}
H = 14*3600
def correr_ait(delay=None, cap=8.0, gain=1.0):
    if delay is None:
        ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow"))
    else:
        ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio", "ait"), q_stock_meas=lambda t: qs0, ratio_nom=R0,
                                 mv_limits={"Q_dil": (0.0, cap*q0)}, ait={"cp_true": pulso, "delay_s": delay, "gain": gain, "R_max": 3*cap*R0})
    return simulate(pl, x0, H, DT, ctrl, disturbance=dist3, log_every=12)

casos = {"sin AIT (dilución fija)": correr_ait(None), "AIT, retardo 0": correr_ait(0.0), "AIT, retardo 15 min": correr_ait(900.0),
         "AIT, retardo 1 h": correr_ait(3600.0), "AIT, retardo 2 h": correr_ait(7200.0),
         "AIT, retardo 0, válvula 3x": correr_ait(0.0, cap=3.0), "AIT, retardo 0, ganancia 0.3": correr_ait(0.0, gain=0.3)}
rows = []
for name, df in casos.items():
    h = df.t_h.values; ppm = df.ppm_ClO4.values; Qd = 3600*df.Qd.values
    rows.append({"caso": name, "ppm pico": ppm.max(), "horas sobre 40 ppm": np.trapezoid((ppm > 40).astype(float), h),
                 "Y mínima %": df.recovery_pct.min(), "Y final %": df.recovery_pct.iloc[-1],
                 "dilución máx [m3/h]": Qd.max(), "agua extra [m3]": np.trapezoid(Qd - 3600*q0, h)})
tabla = pd.DataFrame(rows).set_index("caso"); tabla.round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
cols = ["tab:gray", "tab:blue", "tab:cyan", "tab:orange", "tab:red", "tab:purple", "tab:green"]
for (name, df), col in zip(casos.items(), cols):
    ls = "--" if name.startswith("sin") else "-"
    ax[0].plot(df.t_h, df.ppm_ClO4, color=col, ls=ls, label=name)
    ax[1].plot(df.t_h, 3600*df.Qd, color=col, ls=ls)
    ax[2].plot(df.t_h, df.recovery_pct, color=col, ls=ls)
ax[0].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1); ax[0].legend(fontsize=7)
for a in ax: a.axvspan(3, 5, color="orange", alpha=0.12)
ax[0].set(xlabel="t [h]", ylabel="ppm", title="ClO₄⁻ en producto (sombra = pulso en la alimentación)")
ax[1].set(xlabel="t [h]", ylabel="m³/h", title="Agua de dilución (FIRC 3103)"); ax[2].set(xlabel="t [h]", ylabel="%", title="Recuperación de KNO₃")
fig.tight_layout(); fig.savefig("../docs/fig/05_AIT3105_retardo.png"); plt.show()

## Lectura
* **Sin `AIT`:** el pulso apenas supera el límite provisional de 40 ppm, porque el ppm responde lento (la purga arrastra el ClO₄ en horas) y el pulso es corto.
* **Con `AIT` sin retardo:** la compensación total mantiene el ppm, pero **anula la recuperación** mientras dura el pulso y gasta agua adicional del orden de cientos de m³. La compensación total no es una buena ley económica.
* **Retardo:** a 15 min casi no cambia nada; con retardos de una a dos horas (comparables con el pulso) el agua llega cuando el pulso ya pasó: se gasta la misma agua y el ppm pico vuelve a lo del caso sin analizador. **El retardo importa si el pulso es corto frente a él**, por eso hay que definirlo.
* **Válvula de 3×:** limita el daño a la recuperación pero también la compensación.
* **Ganancia parcial:** es la palanca para negociar ppm contra recuperación; su valor debe salir de una evaluación económica (pérdida por producto fuera de norma frente a producción perdida).

**No se modela:** ruido y deriva del analizador, la ubicación exacta del punto de muestreo, tiempos muertos de transporte entre el analizador y la válvula, ni lógica de alarmas (AHL). Faltan por definir: retardo de `AIT 3105`, ganancia y límites de `FFIC 3102`, capacidad real de la válvula de `FIRC 3103`.